## 議員データ収集

In [2]:
import bs4
import requests
import re
from urllib.parse import urljoin
import os
from params.paths import ROOT_DIR
import pandas as pd
import time
from tqdm import tqdm
import json

from file_handling.file_read_writer import write_json, read_json
from collections import Counter
from api_requests.prompter import DeepResearchGemini

SHUGIIN_REPR_URL = 'https://kokkai.sugawarataku.net/giin/rgiin.html'
SANGIIN_REPR_URL = 'https://kokkai.sugawarataku.net/giin/cgiin.html'

LOWER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list')
LOWER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(LOWER_HOUSE_DATA_DIR, 'historical')
UPPER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_sangiin', 'repr_list')
UPPER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(UPPER_HOUSE_DATA_DIR, 'historical')
LOWER_HOUSE_LOG = os.path.join(LOWER_HOUSE_DATA_DIR, 'log.txt')
UPPER_HOUSE_LOG = os.path.join(UPPER_HOUSE_DATA_DIR, 'log.txt')
os.makedirs(LOWER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)
os.makedirs(UPPER_HOUSE_DATA_HISTORICAL_TMP, exist_ok=True)

In [2]:
def get_repr_data_prompt(name):
	prompt =  """\
			この政治家のデータを集めてほしいです。
			政治家の名前はNAMEです。
			返答フォーマットは以下のようにしてください。
			
			{
			"name_kanji": "名前",
			"name_kana": "名前のふりがな",
			"years": [
			"当選年-当選月-当選日",
			"当選年-当選月-当選日",
			...
			],
			"election_data": [
				{
					"year": "当選年",
					"month": "当選月",
					"day": "当選日",
					"election_name": "当選回次",
					"district": "選挙区",
					"party": "政党",
					"result": "当選",
					"election_freq": "（1回目）"
				},
				...
			]
			}

			返答フォーマットの例：
			{
				"name_kanji": "七条明",
				"name_kana": "しちじょうあきら",
				"years": [
					"1993-07-18",
					...
				],
				"election_data": [
					{
						"year": "1993年",
						"month": "7月",
						"day": "18日",
						"election_name": "第40回衆議院議員総選挙",
						"district": "徳島全県区",
						"party": "自由民主党",
						"result": "当選",
						"election_freq": "（1回目）"
					},
					...
				]
			}
	"""
	prompt = prompt.replace("NAME", name)

	return prompt

research_system_prompt = """\
	あなたは政治家のデータを集めるアシスタントです。常に最新のデータを集めるために必ず検索をすることを心がけてください。
	また、同姓同名の政治家にも気を付けてください。もし同姓同名政治家がいた場合、これから与える返答フォーマットは無視してください。
	その際は「同姓同名の政治家がいる可能性があります」と返答してください。ちなみに地方政治家は含めず、国会議員のみを集めてください。
	jsonを回答する際には余計な説明等をつけず、生のjsonのみを返答してください。コードブロック（```json）をつけることは絶対にしないでください。
	あなたの返答フォーマットはそのままjson.loadsでパースできるようにしてください。
	"""


In [ ]:
class ReprHistoricalDataCollector:
	def __init__(self):
		self.prompter = DeepResearchGemini(model_name="gemini-2.5-pro")



	def process_one_election_row(self, row):

		def try_with_backup(cls1, cls2):
			attempt1 = row.find('div', {'class': cls1})
			if attempt1:
				return attempt1.text
			attempt2 = row.find('div', {'class': cls2})
			if attempt2:
				return attempt2.text
			return ""
			
		try:
			
			year = row.find('div', {'class': 'el1'}).text
			month = row.find('div', {'class': 'el2'}).text
			day = row.find('div', {'class': 'el3'}).text
			election_name = row.find('div', {'class': 'el4'}).text
			district = try_with_backup('el5', 'elc5')
			party = try_with_backup('el6', 'elc6')
			result = try_with_backup('el7', 'elc7')
			election_freq = row.find('div', {'class': 'el8'}).text
		except Exception as e:
			print(f'Error processing row: {row},\n error: {e}')
			raise e
		return {'year': year, 'month': month, 'day': day, 'election_name': election_name, 'district': district, 'party': party, 'result': result, 'election_freq': election_freq}

	def retrieve_info_of_one_repr(self, url):
		html = requests.get(url).content
		soup = bs4.BeautifulSoup(html, 'html.parser')
		repr_data = soup.find_all('div', {'class':'jt2'})
		name_kanji = repr_data[0].text
		name_kana = repr_data[1].text
		years = re.findall(r"\d{4}/\d{2}/\d{2}", repr_data[3].text)
		years = [year.replace('/', '-') for year in years]

		election_data = soup.find_all('div', {'class':'em1'})
		election_data = [self.process_one_election_row(row) for row in election_data]

		return {'name_kanji': name_kanji, 'name_kana': name_kana, 'years': years, 'election_data': election_data}

	def collect(self, house:str):
		if house == 'upper':
			url = SANGIIN_REPR_URL
			log_path = UPPER_HOUSE_LOG
		elif house == 'lower':
			url = SHUGIIN_REPR_URL
			log_path = LOWER_HOUSE_LOG
		tempDir = UPPER_HOUSE_DATA_HISTORICAL_TMP if house == 'upper' else LOWER_HOUSE_DATA_HISTORICAL_TMP
		houseDir = UPPER_HOUSE_DATA_DIR if house == 'upper' else LOWER_HOUSE_DATA_DIR
		print(f'Collecting historical data for {house} house representatives into {tempDir}')
		resp = requests.get(url)
		resp.encoding = "cp932"
		soup = bs4.BeautifulSoup(resp.text, 'lxml')
		links = soup.find_all('span', {'class':'zt5'})
		print(links)
		names = [link.find('a').get_text(strip=True) for link in links]
		print(names)
		hrefs = [link.find('a').get('href') for link in links]
		if Counter(names).most_common()[0][1] > 1:
			print(Counter(names).most_common())
			raise ValueError('There are duplicate names in the historical data')

		if os.path.exists(log_path):
			with open(log_path, 'r') as f:
				done_names = f.readlines()
		else:
			done_names = []


		for idx, (name, href) in enumerate(zip(names, hrefs)):
			if name in done_names:
				print(f'{name} already done')
				continue
			print(f'Processing {name}-{idx/len(names)*100:.2f}%')
			repr_path = os.path.join(tempDir, f'{name}.json')
			user_input = None
			if os.path.exists(repr_path):
				repr_data = read_json(repr_path)
				if repr_data["name_kana"] != "":
					print(f'{name} already exists')
					with open(log_path, 'a') as f:
						f.write(f'{name}\n')
					continue
				else:
					print(f"Data incomplete for {name}")
					prompt = get_repr_data_prompt(name)
					count = 0
					while True:
						try:
							reply, _, _ = self.prompter.prompt(prompt, research_system_prompt)
							print("REPLY:", reply)
							if reply == "同姓同名の政治家がいる可能性があります":
								# user_input = input(f"同姓同名の政治家がいる可能性があります。{name}の生年月日を入力してください。そっちをまず収集します。")
								user_input = "skip"
								print("SKIPPING")
								break
								
								# reply, _, _ = self.prompter.prompt(
								# 	prompt + "\n" + user_input+"が生年月日の政治家のほうの情報を収集してください。", research_system_prompt
								# )
								# print("REPLY:", reply)
							repr_data = json.loads(reply)
							break
						except Exception as e:
							print(f"Error parsing reply: {e}")
							time.sleep(1)
				if user_input == "skip":
					continue
			else:
				repr_link = urljoin(url, href)
				repr_data = self.retrieve_info_of_one_repr(repr_link)
				
				if repr_data["name_kanji"] != name:
					print(f'{name} has a different name in the historical data {repr_data["name_kanji"]}')
					count = 0
					while True:
						try:
							prompt = get_repr_data_prompt(name)
							reply, _, _ = self.prompter.prompt(prompt, research_system_prompt)
							if reply == "同姓同名の政治家がいる可能性があります":
								user_input = input(f"同姓同名の政治家がいる可能性があります。{name}の生年月日を入力してください。そっちをまず収集します。")
								reply, _, _ = self.prompter.prompt(
									prompt + "\n" + user_input+"が生年月日の政治家のほうの情報を収集してください。", research_system_prompt
								)
							print("REPLY:", reply)
							repr_data = json.loads(reply)
							break
						except Exception as e:
							print(f"Error parsing reply: {e}")
							time.sleep(1)
							count += 1
							if count > 3:
								raise e
					repr_data = json.loads(reply)
			path = os.path.join(tempDir, f'{name}{user_input if user_input else ""}.json')
			write_json(repr_data, path)
			with open(log_path, 'a') as f:
				f.write(f'{name}\n')
			time.sleep(1)

		all_repr_data = []
		if Counter(list(os.listdir(tempDir))).most_common()[0][1] > 1:
			print(Counter(list(os.listdir(tempDir))).most_common())
			raise ValueError('There are duplicate names in the historical data')

		covered_files = set()
		for repr_file in os.listdir(tempDir):
			if repr_file in covered_files:
				raise ValueError(f'{repr_file} already exists')
			covered_files.add(repr_file)
			repr_path = os.path.join(tempDir, repr_file)
			repr_data = read_json(repr_path)
			all_repr_data.append(repr_data)
		all_repr_data_names = [repr_data['name_kanji'] for repr_data in all_repr_data]
		if Counter(all_repr_data_names).most_common()[0][1] > 1:
			print(Counter(all_repr_data_names).most_common())
			raise ValueError('There are duplicate names in the historical data')

		write_json({'data':all_repr_data}, os.path.join(houseDir, 'historical.json'))

In [ ]:
sc = ReprHistoricalDataCollector()
# sc.collect("lower")
sc.collect("upper")

[<span class="zt5"><a href="r00893.html">相川勝六</a></span>, <span class="zt5"><a href="r01920.html">逢沢一郎</a></span>, <span class="zt5"><a href="r00633.html">逢沢寛</a></span>, <span class="zt5"><a href="r01444.html">合沢栄</a></span>, <span class="zt5"><a href="r01360.html">相沢武彦</a></span>, <span class="zt5"><a href="r01651.html">逢沢英雄</a></span>, <span class="zt5"><a href="r01648.html">相沢英之</a></span>, <span class="zt5"><a href="r01560.html">愛知和男</a></span>, <span class="zt5"><a href="r00977.html">愛知揆一</a></span>, <span class="zt5"><a href="r01542.html">愛野興一郎</a></span>, <span class="zt5"><a href="r00875.html">愛野時一郎</a></span>, <span class="zt5"><a href="r02781.html">相原しの</a></span>, <span class="zt5"><a href="r02542.html">青木愛</a></span>, <span class="zt5"><a href="r00203.html">青木清左衛門</a></span>, <span class="zt5"><a href="r00261.html">青木孝義</a></span>, <span class="zt5"><a href="r02155.html">青木宏之</a></span>, <span class="zt5"><a href="r00528.html">青木正</a></span>, <span class="zt5"><a href="r01

KeyboardInterrupt: 

In [20]:
LOWER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list')
LOWER_HOUSE_TARGET_DIR = os.path.join(LOWER_HOUSE_DATA_DIR, 'current')
UPPER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_sangiin', 'repr_list')
UPPER_HOUSE_TARGET_DIR = os.path.join(UPPER_HOUSE_DATA_DIR, 'current')
LOWER_HOUSE_SOURCE_PATH = os.path.join(LOWER_HOUSE_DATA_DIR, "20250911_repr_list.json")
UPPER_HOUSE_SOURCE_PATH = os.path.join(UPPER_HOUSE_DATA_DIR, "20250911_repr_list.json")

current_repr_list_upper = read_json(UPPER_HOUSE_SOURCE_PATH)['reprs']
current_repr_list_lower = read_json(LOWER_HOUSE_SOURCE_PATH)['reprs']
print(current_repr_list_upper)
print(current_repr_list_lower)

os.makedirs(LOWER_HOUSE_TARGET_DIR, exist_ok=True)
os.makedirs(UPPER_HOUSE_TARGET_DIR, exist_ok=True)



{'立憲': [{'name': '青木   愛', 'yomikata': 'あおき あい', 'kaiha': '立憲', 'district': '比例', 'period': '令和10年7月25日', 'link': 'https://www.sangiin.go.jp/japanese/joho1/kousei/giin/profile/7007006.htm'}, {'name': '石垣 のりこ [小川 のり子]', 'yomikata': 'いしがき のりこ', 'kaiha': '立憲', 'district': '宮城', 'period': '令和13年7月28日', 'link': 'https://www.sangiin.go.jp/japanese/joho1/kousei/giin/profile/7019003.htm'}, {'name': '石橋  通宏', 'yomikata': 'いしばし みちひろ', 'kaiha': '立憲', 'district': '比例', 'period': '令和10年7月25日', 'link': 'https://www.sangiin.go.jp/japanese/joho1/kousei/giin/profile/7010007.htm'}, {'name': '泉   房穂', 'yomikata': 'いずみ ふさほ', 'kaiha': '立憲', 'district': '兵庫', 'period': '令和13年7月28日', 'link': 'https://www.sangiin.go.jp/japanese/joho1/kousei/giin/profile/7025007.htm'}, {'name': '打越 さく良 [村木 さく良]', 'yomikata': 'うちこし さくら', 'kaiha': '立憲', 'district': '新潟', 'period': '令和13年7月28日', 'link': 'https://www.sangiin.go.jp/japanese/joho1/kousei/giin/profile/7019006.htm'}, {'name': '小沢  雅仁', 'yomikata': 'おざわ まさひと', 'kaiha':

## Now do the same for the current repr list

In [34]:
from dotenv import load_dotenv
import psycopg2
from collections import Counter

load_dotenv()


def get_all_politicians_from_db(cur: psycopg2.extensions.cursor):
    cur.execute("SELECT name_kanji FROM person")
    return [row[0] for row in cur.fetchall()]



conn = None
try:
    conn = psycopg2.connect(
        dbname="kokkaidoc",
        user="postgres",
        password=os.getenv("PSQL_DATABASE_PASSWORD"),
        host="localhost",
        port="5432",
    )
    print("Connected.")

    with conn.cursor() as cur:
        politicians_in_db = get_all_politicians_from_db(cur)
        print(Counter(politicians_in_db).most_common())
        print('Got', len(politicians_in_db), 'politicians from db')
        
        politicians_in_db = set(politicians_in_db)

    print("Done.")

except Exception as e:
    if conn:
        conn.rollback()
    raise
finally:
    if conn:
        conn.close()

Connected.
[('鈴木強平', 2), ('田中正巳', 2), ('青木愛', 2), ('樽井良和', 2), ('玉置一弥', 2), ('久保等', 2), ('釘宮磐', 2), ('八代英太', 2), ('神田博', 2), ('岩動道行', 2), ('佐藤謙一郎', 2), ('石原健太郎', 2), ('木村守江', 2), ('相沢武彦', 2), ('岸信夫', 2), ('鴻池祥肇', 2), ('木村義雄', 2), ('佐藤ゆかり', 2), ('浜田卓二郎', 2), ('沓掛哲男', 2), ('長浜博行', 2), ('沢田政治', 2), ('渡部通子', 2), ('小坂憲次', 2), ('石井桂', 2), ('辻政信', 2), ('前川清成', 2), ('自見庄三郎', 2), ('石井章', 2), ('桜内義雄', 2), ('遠山清彦', 2), ('最上英子', 2), ('大木浩', 2), ('荒井広幸', 2), ('和田博雄', 2), ('久保田藤麿', 2), ('肥田美代子', 2), ('山花秀雄', 2), ('佐藤公治', 2), ('河口陽一', 2), ('藤井孝男', 2), ('井上知治', 2), ('浅尾慶一郎', 2), ('中村勇太', 2), ('大久保直彦', 2), ('堂森芳夫', 2), ('小池百合子', 2), ('鈴木宗男', 2), ('松沢兼人', 2), ('大石正光', 2), ('帆足計', 2), ('山下春江', 2), ('鬼木勝利', 2), ('糸山英太郎', 2), ('中村哲治', 2), ('藤岡隆雄', 2), ('島尻安伊子', 2), ('佐藤泰介', 2), ('大石尚子', 2), ('世耕政隆', 2), ('浮島智子', 2), ('秋元司', 2), ('石原慎太郎', 2), ('江田三郎', 2), ('臼井荘一', 2), ('北村哲男', 2), ('加藤進', 2), ('馳浩', 2), ('玉置和郎', 2), ('松浦定義', 2), ('桜井奎夫', 2), ('弘友和夫', 2), ('若松謙維', 2), ('井村徳二', 2), ('笹森順造', 2), ('苫米地義三', 2), 

In [29]:
from utils.string_process import clean_repr_name

def get_repr_data_prompt(name):
	prompt =  """\
			この政治家のデータを集めてほしいです。
			政治家の名前はNAMEです。
			返答フォーマットは以下のようにしてください。
			
			{
			"name_kanji": "名前",
			"name_kana": "名前のふりがな",
			"years": [
			"当選年-当選月-当選日",
			"当選年-当選月-当選日",
			...
			],
			"election_data": [
				{
					"year": "当選年",
					"month": "当選月",
					"day": "当選日",
					"election_name": "当選回次",
					"district": "選挙区",
					"party": "政党",
					"result": "当選",
					"election_freq": "（1回目）"
				},
				...
			]
			}

			返答フォーマットの例：
			{
				"name_kanji": "七条明",
				"name_kana": "しちじょうあきら",
				"years": [
					"1993-07-18",
					...
				],
				"election_data": [
					{
						"year": "1993年",
						"month": "7月",
						"day": "18日",
						"election_name": "第40回衆議院議員総選挙",
						"district": "徳島全県区",
						"party": "自由民主党",
						"result": "当選",
						"election_freq": "（1回目）"
					},
					...
				]
			}
	"""
	prompt = prompt.replace("NAME", name)

	return prompt

research_system_prompt = """\
	あなたは政治家のデータを集めるアシスタントです。常に最新のデータを集めるために必ず検索をすることを心がけてください。
	また、同姓同名の政治家にも気を付けてください。もし同姓同名政治家がいた場合、これから与える返答フォーマットは無視してください。
	その際は「同姓同名の政治家がいる可能性があります」と返答してください。ちなみに地方政治家は含めず、国会議員のみを集めてください。
	jsonを回答する際には余計な説明等をつけず、生のjsonのみを返答してください。コードブロック（```json）をつけることは絶対にしないでください。
	あなたの返答フォーマットはそのままjson.loadsでパースできるようにしてください。
	"""

def iterate_current_repr_list(repr_list):
	for party, reprs in repr_list.items():
		for repr in reprs:
			yield repr



In [33]:
from api_requests.prompter import DeepResearchGemini
prompter = DeepResearchGemini(model_name="gemini-2.5-pro")
from file_handling.file_read_writer import write_json

for repr_list, target_dir in zip([current_repr_list_upper, current_repr_list_lower], [UPPER_HOUSE_TARGET_DIR, LOWER_HOUSE_TARGET_DIR]):
	for repr in iterate_current_repr_list(repr_list):
		name = clean_repr_name(repr['name'])
		if name in politicians_in_db:
			# skipping repr names that already exist in db
			continue
		if os.path.exists(os.path.join(target_dir, f'{name}.json')):
			# skipping repr names that already exist in target dir
			continue
		count = 0
		while True:
			try:
				prompt = get_repr_data_prompt(name)
				reply, _, _ = prompter.prompt(prompt, research_system_prompt)
				print("GOT REPLY")
				print(reply)
				reply = reply.replace("```json", "").replace("```", "")
				repr_data = json.loads(reply)
				write_json(repr_data, os.path.join(target_dir, f'{name}.json'))
				break
			except Exception as e:
				print(f"Error processing {name}: {e}")
				time.sleep(1)
				count += 1
				if count > 3:
					raise e
			continue
		
		


PROMPTING GEMINI
	あなたは政治家のデータを集めるアシスタントです。常に最新のデータを集めるために必ず検索をすることを心がけてください。
	また、同姓同名の政治家にも気を付けてください。もし同姓同名政治家がいた場合、これから与える返答フォーマットは無視してください。
	その際は「同姓同名の政治家がいる可能性があります」と返答してください。ちなみに地方政治家は含めず、国会議員のみを集めてください。
	jsonを回答する際には余計な説明等をつけず、生のjsonのみを返答してください。コードブロック（```json）をつけることは絶対にしないでください。
	あなたの返答フォーマットはそのままjson.loadsでパースできるようにしてください。
	


			この政治家のデータを集めてほしいです。
			政治家の名前は自見はなこです。
			返答フォーマットは以下のようにしてください。

			{
			"name_kanji": "名前",
			"name_kana": "名前のふりがな",
			"years": [
			"当選年-当選月-当選日",
			"当選年-当選月-当選日",
			...
			],
			"election_data": [
				{
					"year": "当選年",
					"month": "当選月",
					"day": "当選日",
					"election_name": "当選回次",
					"district": "選挙区",
					"party": "政党",
					"result": "当選",
					"election_freq": "（1回目）"
				},
				...
			]
			}

			返答フォーマットの例：
			{
				"name_kanji": "七条明",
				"name_kana": "しちじょうあきら",
				"years": [
					"1993-07-18",
					...
				],
				"election_data": [
					{
						"year": "1993年",
						"month": "7月",
						"day": "18日",
						"election_name